# Flujo de Calor Global
***
created: 08/05/2026                 updated: ------

Estas celdas de código buscan explorar los distintos archivos que contienen los flujos de calor y calcular el flujo de calor global. Con esto, se crea un archivo csv que tenga por columans: cuenca, años, resolución, si es robusta y flujo de calor.

In [1]:
# Packages for calculations and data manipulation
import numpy as np
import xarray as xr
import pandas as pd
from surface import surface
# For progress bar
from tqdm import tqdm

# For files manipulation
import os
import glob

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

### Archivo de muestra

In [2]:
ds_muestra = xr.open_dataset('./Data/Heat_flux/Heat_flux_1990_2025_25_4k_robust.nc')
ds_muestra

<xarray.Dataset> Size: 2GB
Dimensions:          (latitude: 563, longitude: 1355, pressure: 125)
Coordinates:
  * latitude         (latitude) float64 5kB -77.75 -77.5 -77.25 ... 63.0 63.25
  * longitude        (longitude) float64 11kB -180.0 -179.8 ... 179.8 180.0
    mask             (latitude, longitude) float64 6MB ...
    basin            (latitude, longitude) <U18 55MB ...
    batimetry        (latitude, longitude) float64 6MB ...
    surface          (latitude, longitude) float64 6MB ...
    heat_flux        (latitude, longitude) float64 6MB ...
    heat_flux_error  (latitude, longitude) float64 6MB ...
  * pressure         (pressure) int64 1kB 4000 4020 4040 4060 ... 6440 6460 6480
Data variables:
    tendency         (latitude, longitude, pressure) float64 763MB ...
    density          (latitude, longitude, pressure) float64 763MB ...
    Cp               (latitude, longitude, pressure) float64 763MB ...
Attributes:
    description:  Dataset of temperature tendencies by pressure levels betwee...

In [3]:
ds_muestra.close()

### Calculo de Flujo de Calor Global

In [4]:
# Function for global calculation
def flujo_glob(ds):
    cuencas = np.unique(ds.basin.values)

    basin_name = [] # Basin name
    basin_surface = [] # Total basin surface
    basin_Q = [] # Heat flux on basin
    basin_Q_std = []

    for cuenca in tqdm(cuencas, leave= False):
        if str(cuenca) == '' or str(cuenca) == 'nan' or pd.isna(cuenca):
            continue

        else:
            mask = (ds.basin == cuenca)
            ds_basin = ds.where(mask)

            ds_basin = ds_basin.assign_coords(
                surface=ds.surface.where(mask),
                batimetry=ds.batimetry.where(mask),
                heat_flux=ds.heat_flux.where(mask),
                heat_flux_error = ds.heat_flux_error.where(mask))


            area = surface(ds_basin, 4000)
            Q = ds_basin.heat_flux.mean().values
            Q_std = ds_basin.heat_flux_error.mean().values

            basin_name.append(cuenca)
            basin_surface.append(area)
            basin_Q.append(Q)
            basin_Q_std.append(Q_std)
    
    q_glob = 0
    q_glob_std = 0

    for i in range(len(basin_name)):
        if not np.isnan(basin_Q[i]):
            q_glob += basin_surface[i] * basin_Q[i]
            q_glob_std += (basin_surface[i] * basin_Q_std[i] / 20)

    area_glob = np.nansum(basin_surface)

    Q_glob = q_glob / area_glob
    Q_glob_std = (2 * np.sqrt(q_glob_std)) / area_glob

    return (Q_glob, Q_glob_std, basin_name, basin_Q, basin_Q_std, basin_surface)

In [5]:
# Directory for general rute
dir = './Data/Heat_flux/'

files = glob.glob(os.path.join(dir, '*.nc'))

# Listo for rows in the Dataframe
datos_csv = []

# Iterate for every file
for file in files:
    file_name = os.path.basename(file)

    name_parts = file_name.replace('.nc', '').split('_')

    # Extract information from the elements
    years = f'{name_parts[2]}-{name_parts[3]}'
    resolution = name_parts[4]

    if resolution == '25':
        resolution = 0.25
    elif resolution == '5':
        resolution = 0.5

    robusta = 'robust' in name_parts

    # Open dataset and calculate flux
    ds = xr.open_dataset(file)
    Q_glob, Q_glob_std, basin_name, basin_Q , basin_Q_std, basin_surface= flujo_glob(ds)

    # Add to the csv
    datos_csv.append({
        'Basin': 'Global',
        'Years': years,
        'Resolution' : resolution,
        'Robust_tendency': robusta,
        'Surface' : f'{np.nansum(basin_surface):.2f}',
        'Heat_flux': f'{Q_glob:.3f}',
        'Heat_flux_error' : f'{Q_glob_std:.3f}'
    })

    # Insert every basin
    for cuenca, q_cuenca, q_std_cuenca, surface_cuenca in zip(basin_name, basin_Q, basin_Q_std, basin_surface):
        datos_csv.append({
            'Basin' : str(cuenca),
            'Years' : years,
            'Resolution' : resolution,
            'Robust_tendency': robusta,
            'Surface' : f'{surface_cuenca:.2f}',
            'Heat_flux': f'{q_cuenca:.2f}',
            'Heat_flux_error' : f'{q_std_cuenca:.2f}'
        })
    
    ds.close()

    # Divide line
    datos_csv.append({
        'Basin' : '----------------',
        'Years' : '---------',
        'Resolution' : '----------',
        'Tendency_type': '-------------',
        'Heat_flux': '---------'
    })


df = pd.DataFrame(datos_csv)

  0%|          | 0/29 [00:00<?, ?it/s]

In [6]:
save_dir = './HeatGlobalFlux.csv'
df.to_csv(save_dir, index = False)